# Explorative Analyse: Arbeitsmarkt Regensburg (Jobcenter) — Teil 10: Rechtskreisvergleich SGB II vs. SGB III

Letzter offener Punkt aus [`ArbeitslosenAnalyse.md`](../MD/ArbeitslosenAnalyse.md): „Vergleich ‚Insgesamt (SGB II+SGB III)‘ vs. ‚SGB II‘ — wie hoch ist der Jobcenter-Anteil an der gesamten Arbeitslosigkeit, und verändert er sich über die Zeit?“

Alle bisherigen Notebooks haben ausschließlich den **Rechtskreis SGB II** (Jobcenter-Zuständigkeit) betrachtet. Das Blatt „1.1 Eckwerte“ enthält daneben aber auch die Spalte **„Insgesamt (SGB II und SGB III)“** — die Summe aus Jobcenter- und Agentur-für-Arbeit-Zuständigkeit. Die Differenz ergibt den **SGB-III-Anteil** (Arbeitslosenversicherung, Agentur für Arbeit).

**Keine neuen Rohdaten nötig:** Beide Spalten (`insgesamt` und `sgb2`) sind bereits in [`eckwerte_long.csv`](../Data/processed/eckwerte_long.csv) enthalten — dieses Notebook liest ausschließlich diese bereits aufbereitete Datei, genau wie [`02_Analyse_SGBII.ipynb`](02_Analyse_SGBII.ipynb). Keine Wartezeit-Problematik, volle 18 Monate Januar 2025 – Juni 2026.

In [1]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

PROCESSED_DIR = Path("..") / "Data" / "processed"

df_long = pd.read_csv(PROCESSED_DIR / "eckwerte_long.csv")
df_long["period"] = pd.PeriodIndex(df_long["period"], freq="M")

merkmale = [
    ("Arbeitslose", "Bestand", "Arbeitslose (Bestand)"),
    ("Arbeitslose", "Arbeitslosenquote", "Arbeitslosenquote (%)"),
    ("Arbeitslose", "Langzeitarbeitslose", "Langzeitarbeitslose"),
    ("Unterbeschäftigung", "Unterbeschäftigungsquote", "Unterbeschäftigungsquote (%)"),
]

vergleich = pd.DataFrame(index=pd.PeriodIndex([], freq="M"))
for kategorie, merkmal, label in merkmale:
    sub = df_long[(df_long["kategorie"] == kategorie) & (df_long["merkmal"] == merkmal)]
    sub = sub.set_index("period").sort_index()
    vergleich[f"{label}: Insgesamt"] = sub["insgesamt"]
    vergleich[f"{label}: SGB II"] = sub["sgb2"]
    vergleich[f"{label}: SGB III"] = sub["insgesamt"] - sub["sgb2"]
    vergleich[f"{label}: Jobcenter-Anteil (%)"] = (sub["sgb2"] / sub["insgesamt"] * 100).round(1)

vergleich.tail()

,Arbeitslose (Bestand): Insgesamt,Arbeitslose (Bestand): SGB II,Arbeitslose (Bestand): SGB III,Arbeitslose (Bestand): Jobcenter-Anteil (%),Arbeitslosenquote (%): Insgesamt,Arbeitslosenquote (%): SGB II,Arbeitslosenquote (%): SGB III,Arbeitslosenquote (%): Jobcenter-Anteil (%),Langzeitarbeitslose: Insgesamt,Langzeitarbeitslose: SGB II,Langzeitarbeitslose: SGB III,Langzeitarbeitslose: Jobcenter-Anteil (%),Unterbeschäftigungsquote (%): Insgesamt,Unterbeschäftigungsquote (%): SGB II,Unterbeschäftigungsquote (%): SGB III,Unterbeschäftigungsquote (%): Jobcenter-Anteil (%)
period,,,,,,,,,,,,,,,,
2026-02,4226.0,1542.0,2684.0,36.5,3.6,1.3,2.3,36.1,940.0,662.0,278.0,70.4,4.2,1.7,2.5,40.5
2026-03,4052.0,1543.0,2509.0,38.1,3.5,1.3,2.2,37.1,947.0,677.0,270.0,71.5,4.0,1.7,2.3,42.5
2026-04,3911.0,1565.0,2346.0,40.0,3.3,1.3,2.0,39.4,983.0,694.0,289.0,70.6,3.9,1.8,2.1,46.2
2026-05,3835.0,1552.0,2283.0,40.5,3.2,1.3,1.9,40.6,1011.0,714.0,297.0,70.6,3.9,1.7,2.2,43.6
2026-06,3774.0,1500.0,2274.0,39.7,3.2,1.3,1.9,40.6,996.0,709.0,287.0,71.2,3.8,1.7,2.1,44.7


## 1. Jobcenter-Anteil am Arbeitslosenbestand insgesamt

Von allen Arbeitslosen in Regensburg fällt nur ein Teil in die Zuständigkeit des Jobcenters (Rechtskreis SGB II) — der Rest liegt bei der Agentur für Arbeit (Rechtskreis SGB III, z. B. Empfänger von Arbeitslosengeld I).

In [2]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=vergleich.index.to_timestamp(), y=vergleich["Arbeitslose (Bestand): SGB II"], name="SGB II (Jobcenter)",
    mode="lines", stackgroup="bestand", line=dict(color="#2563eb", width=0.5),
))
fig.add_trace(go.Scatter(
    x=vergleich.index.to_timestamp(), y=vergleich["Arbeitslose (Bestand): SGB III"], name="SGB III (Agentur für Arbeit)",
    mode="lines", stackgroup="bestand", line=dict(color="#7c3aed", width=0.5),
))
fig.add_trace(go.Scatter(
    x=vergleich.index.to_timestamp(), y=vergleich["Arbeitslose (Bestand): Jobcenter-Anteil (%)"], name="Jobcenter-Anteil (%)",
    mode="lines+markers", line=dict(color="#f59e0b", dash="dot"), marker=dict(size=5), yaxis="y2",
))
fig.update_layout(
    title="Arbeitslose (Bestand): SGB II vs. SGB III",
    yaxis=dict(title="Personen"),
    yaxis2=dict(title="Jobcenter-Anteil (%)", overlaying="y", side="right", ticksuffix="%"),
    xaxis_title="Monat", hovermode="x unified", legend=dict(orientation="h", y=1.15), height=460,
)
fig

Der Jobcenter-Anteil (SGB II) an allen Arbeitslosen in Regensburg liegt stabil bei rund 40 % (39,3 % im Januar 2025, 39,7 % im Juni 2026, Spanne 36,3–42,2 %, Ø 39,8 %) — **die Mehrheit der Arbeitslosen (rund 60 %) fällt also in die Zuständigkeit der Agentur für Arbeit (SGB III)**, nicht des Jobcenters. Absolut ist die SGB-III-Zahl mit 2.273 → 2.274 Personen ebenfalls deutlich größer als die SGB-II-Zahl (1.470 → 1.500). Der Anteil selbst ist über die 18 Monate bemerkenswert stabil geblieben, trotz der in Kapitel 3 unten sichtbaren gegenläufigen Entwicklung bei der Langzeitarbeitslosigkeit.

## 2. Langzeitarbeitslosigkeit: konzentriert sich das stärker im SGB II?

Naheliegende Vermutung: Arbeitslosengeld I (SGB III) ist zeitlich befristet (i. d. R. 12–24 Monate) — Langzeitarbeitslosigkeit müsste sich daher überproportional im SGB II konzentrieren.

In [3]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=vergleich.index.to_timestamp(), y=vergleich["Langzeitarbeitslose: SGB II"], name="SGB II (Jobcenter)",
    mode="lines", stackgroup="lza", line=dict(color="#2563eb", width=0.5),
))
fig.add_trace(go.Scatter(
    x=vergleich.index.to_timestamp(), y=vergleich["Langzeitarbeitslose: SGB III"], name="SGB III (Agentur für Arbeit)",
    mode="lines", stackgroup="lza", line=dict(color="#7c3aed", width=0.5),
))
fig.add_trace(go.Scatter(
    x=vergleich.index.to_timestamp(), y=vergleich["Langzeitarbeitslose: Jobcenter-Anteil (%)"], name="Jobcenter-Anteil (%)",
    mode="lines+markers", line=dict(color="#f59e0b", dash="dot"), marker=dict(size=5), yaxis="y2",
))
fig.update_layout(
    title="Langzeitarbeitslose: SGB II vs. SGB III",
    yaxis=dict(title="Personen"),
    yaxis2=dict(title="Jobcenter-Anteil (%)", overlaying="y", side="right", ticksuffix="%"),
    xaxis_title="Monat", hovermode="x unified", legend=dict(orientation="h", y=1.15), height=460,
)
fig

Die Vermutung bestätigt sich deutlich: Der Jobcenter-Anteil an den Langzeitarbeitslosen liegt bei 71,2–75,9 % (Ø 72,3 %) — **fast drei Viertel aller Langzeitarbeitslosen in Regensburg fallen in die Zuständigkeit des Jobcenters**, deutlich mehr als der rund 40-prozentige Anteil beim Arbeitslosenbestand insgesamt (Kapitel 1). Das ist plausibel: Arbeitslosengeld I (SGB III) ist zeitlich befristet, sodass Personen nach Ablauf des Anspruchs in den SGB-II-Bezug wechseln, wenn sie weiterhin arbeitslos sind — Langzeitarbeitslosigkeit konzentriert sich dadurch strukturell im SGB II. Der Anteil ist zudem leicht rückläufig (73,1 % → 71,2 %) — absolut wächst die SGB-III-Langzeitarbeitslosigkeit (185 → 287 Personen, + 55 %) sogar noch stärker als die SGB-II-Zahl (502 → 709, + 41 %, siehe [`03_Statistik.ipynb`](03_Statistik.ipynb)).

## 3. Quoten im Vergleich: Arbeitslosenquote und Unterbeschäftigungsquote

In [4]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=vergleich.index.to_timestamp(), y=vergleich["Arbeitslosenquote (%): Insgesamt"], name="Arbeitslosenquote: Insgesamt", mode="lines+markers", line=dict(color="#374151"), marker=dict(size=5)))
fig.add_trace(go.Scatter(x=vergleich.index.to_timestamp(), y=vergleich["Arbeitslosenquote (%): SGB II"], name="Arbeitslosenquote: SGB II-Anteil", mode="lines+markers", line=dict(color="#2563eb"), marker=dict(size=5)))
fig.add_trace(go.Scatter(x=vergleich.index.to_timestamp(), y=vergleich["Unterbeschäftigungsquote (%): Insgesamt"], name="Unterbeschäftigungsquote: Insgesamt", mode="lines+markers", line=dict(color="#6b7280", dash="dash"), marker=dict(size=5)))
fig.add_trace(go.Scatter(x=vergleich.index.to_timestamp(), y=vergleich["Unterbeschäftigungsquote (%): SGB II"], name="Unterbeschäftigungsquote: SGB II-Anteil", mode="lines+markers", line=dict(color="#7c3aed", dash="dash"), marker=dict(size=5)))
fig.update_layout(
    title="Arbeitslosen- und Unterbeschäftigungsquote: Insgesamt vs. SGB II-Anteil",
    yaxis_title="Prozent", yaxis_ticksuffix="%",
    xaxis_title="Monat", hovermode="x unified", legend=dict(orientation="h", y=1.15), height=460,
)
fig

Arbeitslosenquote und Unterbeschäftigungsquote zeigen ein ähnliches Bild wie der Bestand: Die Insgesamt-Quoten (3,2 % bzw. 4,0→3,8 %) liegen deutlich über den SGB-II-Anteilsquoten (1,3 % bzw. 1,9→1,7 %) — der SGB-II-Anteil an der Arbeitslosenquote liegt bei Ø 40,3 % (nahezu identisch mit dem Anteil am Bestand, da beide Quoten direkt proportional zum jeweiligen Bestand sind). Bei der Unterbeschäftigungsquote ist der SGB-II-Anteil mit Ø 45,6 % etwas höher, aber spürbar rückläufig (47,5 % → 44,7 %).

## 4. Export der aufbereiteten Daten

Grundlage für einen möglichen weiteren Dashboard-Ausbau.

In [5]:
vergleich.to_csv(PROCESSED_DIR / "rechtskreisvergleich_sgb2.csv")
vergleich.tail()

,Arbeitslose (Bestand): Insgesamt,Arbeitslose (Bestand): SGB II,Arbeitslose (Bestand): SGB III,Arbeitslose (Bestand): Jobcenter-Anteil (%),Arbeitslosenquote (%): Insgesamt,Arbeitslosenquote (%): SGB II,Arbeitslosenquote (%): SGB III,Arbeitslosenquote (%): Jobcenter-Anteil (%),Langzeitarbeitslose: Insgesamt,Langzeitarbeitslose: SGB II,Langzeitarbeitslose: SGB III,Langzeitarbeitslose: Jobcenter-Anteil (%),Unterbeschäftigungsquote (%): Insgesamt,Unterbeschäftigungsquote (%): SGB II,Unterbeschäftigungsquote (%): SGB III,Unterbeschäftigungsquote (%): Jobcenter-Anteil (%)
period,,,,,,,,,,,,,,,,
2026-02,4226.0,1542.0,2684.0,36.5,3.6,1.3,2.3,36.1,940.0,662.0,278.0,70.4,4.2,1.7,2.5,40.5
2026-03,4052.0,1543.0,2509.0,38.1,3.5,1.3,2.2,37.1,947.0,677.0,270.0,71.5,4.0,1.7,2.3,42.5
2026-04,3911.0,1565.0,2346.0,40.0,3.3,1.3,2.0,39.4,983.0,694.0,289.0,70.6,3.9,1.8,2.1,46.2
2026-05,3835.0,1552.0,2283.0,40.5,3.2,1.3,1.9,40.6,1011.0,714.0,297.0,70.6,3.9,1.7,2.2,43.6
2026-06,3774.0,1500.0,2274.0,39.7,3.2,1.3,1.9,40.6,996.0,709.0,287.0,71.2,3.8,1.7,2.1,44.7


## 5. Zusammenfassung

- **Arbeitslose (Bestand):** Jobcenter (SGB II) trägt nur rund 40 % der gesamten Arbeitslosigkeit in Regensburg — die Mehrheit (≈ 60 %) liegt bei der Agentur für Arbeit (SGB III). Der Anteil ist über 18 Monate stabil geblieben.
- **Langzeitarbeitslosigkeit:** deutlich anderes Bild — rund 72 % aller Langzeitarbeitslosen fallen ins SGB II, plausibel durch die zeitliche Befristung von Arbeitslosengeld I. Auffällig: Die SGB-III-Langzeitarbeitslosigkeit wächst sogar noch stärker (+ 55 %) als die SGB-II-Zahl (+ 41 %).
- **Quoten:** Arbeitslosen- und Unterbeschäftigungsquote bestätigen dasselbe Bild — SGB-II-Anteil an der Arbeitslosenquote ≈ 40 %, an der Unterbeschäftigungsquote ≈ 46 % (rückläufig).
- **Kernaussage:** Der Jobcenter-Fokus dieses gesamten Projekts (Rechtskreis SGB II) bildet nur einen Teil der Regensburger Arbeitslosigkeit ab — aber genau den Teil, in dem sich Langzeitarbeitslosigkeit überproportional konzentriert und der daher sozial- und arbeitsmarktpolitisch besonders relevant ist.

Damit sind alle in [`ArbeitslosenAnalyse.md`](../MD/ArbeitslosenAnalyse.md) unter „Noch ungenutzte Zeilen in ‚1.1 Eckwerte‘“ gelisteten Vergleichspunkte abgedeckt. Umsetzung im [Streamlit-Dashboard](../Dashboard/app.py).